# Freddie Mac 2015–2023 Modeling Dataset

This notebook merges each cleaned origination cohort with its eligible 24-month target and validates the resulting one-row-per-loan dataset. The merge preserves the observed event rate; no resampling or class weighting belongs in this data-preparation step.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.freddie_modeling_dataset import (
    LABEL_COLUMNS,
    TARGET_AUDIT_COLUMNS,
    run_modeling_dataset_pipeline,
)

print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

## 1. Define the merge contract

The output contains only target-eligible loans. A loan is joined by both `SOURCE YEAR` and `LOAN IDENTIFIER`, and its cohort must agree across the origination and target artifacts.

In [ ]:
YEARS = list(range(2015, 2024))
SELECTED_HORIZON = 24
EXPECTED_LOANS_PER_YEAR = 50000
EXPECTED_ELIGIBLE_LOANS = 437668

PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

input_paths = [
    PROCESSED_DATA_DIRECTORY
    / f"origination_{year}_clean.parquet"
    for year in YEARS
]
input_paths.append(
    PROCESSED_DATA_DIRECTORY
    / "performance_target_2015_2023_24m.parquet"
)

file_check = pd.DataFrame(
    {
        "path": [str(path) for path in input_paths],
        "exists": [path.exists() for path in input_paths],
    }
)
display(file_check)

missing_paths = file_check.loc[
    ~file_check["exists"],
    "path",
].tolist()

if missing_paths:
    raise FileNotFoundError(
        f"Missing modeling-dataset inputs: {missing_paths}"
    )

## 2. Merge, validate, save, and verify the readback

The pipeline rejects duplicate or unmatched keys, year/cohort disagreement, missing or non-binary labels, an incorrect horizon, event-period inconsistencies, and any row loss relative to the validated target file.

In [ ]:
results = run_modeling_dataset_pipeline(
    years=YEARS,
    input_directory=PROCESSED_DATA_DIRECTORY,
    selected_horizon=SELECTED_HORIZON,
    expected_origination_loans_per_year=(
        EXPECTED_LOANS_PER_YEAR
    ),
)

print("Saved outputs:")
for name, path in results["output_paths"].items():
    print(f"{name}: {path}")

## 3. Reconcile annual row counts and outcomes

In [ ]:
annual_summary = results["annual_summary"].copy()
display(
    annual_summary.drop(
        columns="composite_event_rate"
    ).round(3)
)

assert annual_summary["counts_reconcile"].all()
assert annual_summary["merged_loans"].sum() == (
    EXPECTED_ELIGIBLE_LOANS
)
assert results["validation"][
    "readback_validation_matches"
]
assert results["validation"][
    "readback_annual_summary_matches"
]

## 4. Inspect the final dataset

In [ ]:
modeling_dataset = results["modeling_dataset"]

print("Shape:", modeling_dataset.shape)
print("Duplicate loan/year keys:", modeling_dataset.duplicated(
    subset=["SOURCE YEAR", "LOAN IDENTIFIER"]
).sum())
print("Missing labels:", modeling_dataset[LABEL_COLUMNS].isna().sum().to_dict())
display(modeling_dataset.head())

## Modeling boundary

The event-period and observation-window fields are retained only for target auditing. They contain post-origination information and must not enter the predictor matrix. Feature selection and train/evaluation cohort assignment will be handled in the next modeling step.

In [ ]:
AUDIT_ONLY_COLUMNS = [
    column
    for column in TARGET_AUDIT_COLUMNS
    if column not in LABEL_COLUMNS
]

print("Labels:", LABEL_COLUMNS)
print("Exclude from predictors:", AUDIT_ONLY_COLUMNS)